<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 40%, #16213e 70%, #0f3460 100%); padding: 40px; border-radius: 16px; text-align: center; border: 2px solid #e94560; box-shadow: 0 0 40px rgba(233,69,96,0.4);">
  <h1 style="color: #e94560; font-size: 3em; font-weight: 900; margin: 0; letter-spacing: 3px; text-shadow: 0 0 20px rgba(233,69,96,0.8);">🔍 CRIME PREDICTOR</h1>
  <h3 style="color: #a8b2d8; margin: 10px 0 0 0; font-weight: 300; letter-spacing: 8px; font-size: 1.1em;">MACHINE LEARNING · LOS ANGELES CRIME DATA</h3>
  <hr style="border: 1px solid #e94560; margin: 20px auto; width: 60%; opacity: 0.5;">
  <p style="color: #8892b0; font-size: 1em; margin: 0;">Classifying Violent vs Non-Violent Crime using LAPD Data (2020–Present)</p>
</div>


<div style="background:#0d1117; border-left: 4px solid #e94560; padding: 20px 25px; border-radius: 8px; margin: 10px 0;">
<h2 style="color:#e94560; margin-top:0;">📋 Dataset Description</h2>

This dataset contains <b style="color:#cdd6f4;">real crime incidents in the City of Los Angeles</b> from 2020 to the present.  
Provided by the <b style="color:#cdd6f4;">Los Angeles Police Department (LAPD)</b> as an official government dataset.

- Data is transcribed from original crime reports  
- Location fields may appear as (0°, 0°) when information is missing  
- Addresses shown to the nearest hundred block for privacy  
- Since **March 7, 2024**, LAPD transitioned to a new RMS to comply with FBI's **NIBRS** standard  

**Source:** [Crime Data from 2020 to Present – data.gov](https://catalog.data.gov/dataset/crime-data-from-2020-to-present)
</div>

<div style="background:#0d1117; border-left: 4px solid #7aa2f7; padding: 20px 25px; border-radius: 8px; margin: 10px 0;">
<h3 style="color:#7aa2f7; margin-top:0;">🗂️ Feature Glossary</h3>

| Feature | Description |
|---|---|
| `DR_NO` | Unique crime report number |
| `Date Rptd` | Date when the crime was reported |
| `DATE OCC` | Date when the crime occurred |
| `TIME OCC` | Time when the crime occurred |
| `AREA NAME` | Name of the LAPD police area |
| `Crm Cd Desc` | Description of the crime |
| `Vict Age` | Age of the victim |
| `Vict Sex` | Gender of the victim (M/F) |
| `Vict Descent` | Ethnicity/race of the victim |
| `Premis Cd` | Numeric code of crime location |
| `Weapon Used Cd` | Numeric code of weapon used |
| `Weapon Desc` | Description of weapon used |
| `Status Desc` | Description of case status |
| `Part 1-2` | **Target** – Crime category (1=Violent, 2=Non-Violent) |
| `LAT / LON` | GPS coordinates |
</div>

<div style="background:#0d1117; border-left:4px solid #9ece6a; padding:15px 25px; border-radius:8px;"><h2 style="color:#9ece6a; margin:0;">⚙️ 1 · Imports & Configuration</h2></div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import random
import warnings
import os
import pickle

import folium
from folium.plugins import MarkerCluster

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from scipy.stats import skew

from sklearn.preprocessing import (LabelEncoder, StandardScaler,
                                    RobustScaler, PowerTransformer)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               BaggingClassifier, AdaBoostClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)

# ── Global style ──────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pio.renderers.default = "notebook"

CRIME_RED   = '#e94560'
CRIME_BLUE  = '#7aa2f7'
CRIME_DARK  = '#0d1117'
CRIME_MID   = '#1a1a2e'
ACCENT      = '#bb9af7'

mpl.rcParams.update({
    'figure.facecolor': CRIME_DARK,
    'axes.facecolor':   CRIME_MID,
    'axes.edgecolor':   '#2a2a4a',
    'axes.labelcolor':  '#cdd6f4',
    'xtick.color':      '#8892b0',
    'ytick.color':      '#8892b0',
    'text.color':       '#cdd6f4',
    'grid.color':       '#2a2a4a',
    'grid.linestyle':   '--',
    'axes.titlecolor':  CRIME_RED,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
    'font.family':      'DejaVu Sans',
})

os.makedirs('Files', exist_ok=True)
print("✅ All imports loaded | Files/ directory ready")

<div style="background:#0d1117; border-left:4px solid #e94560; padding:15px 25px; border-radius:8px;"><h2 style="color:#e94560; margin:0;">📂 2 · Load & Understand the Dataset</h2></div>

In [ ]:
df = pd.read_csv('Crime_Data_from_2020_to_Present.csv')
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

<div style="background:#0d1117; border-left:4px solid #9ece6a; padding:15px 25px; border-radius:8px;"><h2 style="color:#9ece6a; margin:0;">🕐 3 · Feature Engineering – Temporal Features</h2></div>

In [ ]:
df['DATE OCC']   = pd.to_datetime(df['DATE OCC'])
df['Date Rptd']  = pd.to_datetime(df['Date Rptd'])

df['day_of_week OCC'] = df['DATE OCC'].dt.day_of_week
df['Year OCC']        = df['DATE OCC'].dt.year
df['Month OCC']       = df['DATE OCC'].dt.month
df['Day OCC']         = df['DATE OCC'].dt.day

df['day_of_week Rptd'] = df['Date Rptd'].dt.day_of_week
df['Year Rptd']        = df['Date Rptd'].dt.year
df['Month Rptd']       = df['Date Rptd'].dt.month
df['Day Rptd']         = df['Date Rptd'].dt.day

df['Hour OCC'] = df['TIME OCC'].apply(lambda x: int(str(int(x)).zfill(4)[:2]))
print("✅ Temporal features extracted")
df[['DATE OCC', 'Year OCC', 'Month OCC', 'Day OCC', 'Hour OCC', 'day_of_week OCC']].head()

<div style="background:#0d1117; border-left:4px solid #f7768e; padding:15px 25px; border-radius:8px;"><h2 style="color:#f7768e; margin:0;">🧹 4 · Handling Missing & Duplicate Values</h2></div>

In [ ]:
print(f"🔁 Duplicate rows: {df.duplicated().sum()}")
df.isna().sum().sort_values(ascending=False).head(13)

In [ ]:
# Drop irrelevant / leaky columns
df = df.drop(['Crm Cd', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4',
              'Cross Street', 'Premis Desc', 'Mocodes'], axis=1)

# Drop rows where essential target-adjacent features are missing
df = df.dropna(subset=['Status', 'Crm Cd 1', 'Premis Cd'])

# Fill remaining nulls
df['Weapon Used Cd'] = df['Weapon Used Cd'].fillna(0)
df['Weapon Desc']    = df['Weapon Desc'].fillna('No Weapon')
df['Vict Descent']   = df['Vict Descent'].fillna('Other')

print("✅ After cleaning:")
print(df.isna().sum().sort_values(ascending=False).head(6))

In [ ]:
# Encode victim sex
print("Unique values in Vict Sex:", df['Vict Sex'].unique())
df['Vict Sex'] = df['Vict Sex'].replace({'M': 0, 'F': 1, 'H': 2, 'X': 3})
print("✅ Vict Sex encoded")

<div style="background:#0d1117; border-left:4px solid #7aa2f7; padding:15px 25px; border-radius:8px;"><h2 style="color:#7aa2f7; margin:0;">✂️ 5 · Train / Test Split</h2></div>

In [ ]:
X = df.drop('Part 1-2', axis=1)
y = df['Part 1-2']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set : {X_train.shape[0]:,} rows")
print(f"Test  set : {X_test.shape[0]:,} rows")
print(f"Target balance (train):\n{y_train.value_counts(normalize=True).round(3)}")

<div style="background:#0d1117; border-left:4px solid #e0af68; padding:15px 25px; border-radius:8px;"><h2 style="color:#e0af68; margin:0;">🔧 6 · Iterative Imputation</h2></div>

In [ ]:
# Fix '-' values in Vict Sex
X_train['Vict Sex'] = X_train['Vict Sex'].replace({'-': np.nan}).astype(float)
X_test['Vict Sex']  = X_test['Vict Sex'].replace({'-': np.nan}).astype(float)

# Impute numeric columns
iter_imputer1 = IterativeImputer(max_iter=15, random_state=42)
num_cols = X_train.select_dtypes(include='number').columns

X_train[num_cols] = iter_imputer1.fit_transform(X_train[num_cols])
X_test[num_cols]  = iter_imputer1.transform(X_test[num_cols])

with open('Files/Iterative_imputer_Age.pkl', 'wb') as f:
    pickle.dump(iter_imputer1, f)

# Round Vict Sex back to int
X_train['Vict Sex'] = X_train['Vict Sex'].round().astype(int)
X_test['Vict Sex']  = X_test['Vict Sex'].round().astype(int)

# Remove invalid Vict Sex values
mask = X_train['Vict Sex'] >= 0
X_train = X_train[mask]
y_train = y_train[mask]

print("✅ Imputation complete")
print(f"Vict Sex distribution:\n{X_train['Vict Sex'].value_counts()}")

<div style="background:#0d1117; border-left:4px solid #bb9af7; padding:15px 25px; border-radius:8px;"><h2 style="color:#bb9af7; margin:0;">📊 7 · Exploratory Data Analysis</h2></div>

In [ ]:
# ── 7.1 Interactive Folium Crime Map ─────────────────────────────────────────
df_map = df.sample(n=5000, random_state=42)

colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
          'cadetblue', 'darkblue', 'darkgreen', 'black', 'gray']

crime_types = df_map['Crm Cd Desc'].unique()
crime_color_map = {crime: random.choice(colors) for crime in crime_types}

m = folium.Map(
    location=[34.05, -118.25],
    zoom_start=10,
    tiles='CartoDB dark_matter'
)
marker_cluster = MarkerCluster().add_to(m)

for idx, row in df_map.iterrows():
    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=5,
        popup=folium.Popup(
            f"<b>Crime:</b> {row['Crm Cd Desc']}<br>"
            f"<b>Area:</b> {row['AREA NAME']}<br>"
            f"<b>Date:</b> {row['DATE OCC']}",
            max_width=250
        ),
        color=crime_color_map[row['Crm Cd Desc']],
        fill=True, fill_opacity=0.7
    ).add_to(marker_cluster)

m.save('crime_map.html')
print("✅ Map saved → crime_map.html")
m

In [ ]:
# ── 7.2 Crime Heatmap: Day of Week × Hour ────────────────────────────────────
pivot = pd.pivot_table(
    df, values='Crm Cd 1',
    index=['day_of_week OCC'], columns=['Hour OCC'],
    aggfunc='count', fill_value=0
)
days_map = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}
pivot.index = pivot.index.map(days_map)

pivot_long = pivot.reset_index().melt(
    id_vars='day_of_week OCC', var_name='Hour', value_name='Crime_Count'
)

fig = px.density_heatmap(
    pivot_long, x='Hour', y='day_of_week OCC', z='Crime_Count',
    color_continuous_scale='Reds',
    title='🔥 Crime Density: Day of Week × Hour of Day',
    labels={'Crime_Count': 'Crime Count', 'day_of_week OCC': 'Day'}
)
fig.update_layout(
    width=950, height=450,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#cdd6f4'),
    title_font=dict(size=18, color='#e94560'),
    coloraxis_colorbar=dict(title='Count', tickfont=dict(color='#cdd6f4'))
)
fig.show()

In [ ]:
# ── 7.3 Crime Hours by Type – Violin Plot ────────────────────────────────────
top_crimes = df['Crm Cd Desc'].value_counts().nlargest(10).index

fig, ax = plt.subplots(figsize=(14, 10))
sns.violinplot(
    data=df[df['Crm Cd Desc'].isin(top_crimes)],
    y='Crm Cd Desc', x='Hour OCC',
    order=top_crimes,
    palette=['#e94560'] * 5 + ['#7aa2f7'] * 5,
    cut=0, ax=ax
)
ax.set_title('Crime Hours Distribution by Type (Top 10)', pad=15)
ax.set_xlabel('Hour of Occurrence')
ax.set_ylabel('')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4 Monthly Crime Trend ──────────────────────────────────────────────────
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')
df['year_month'] = df['DATE OCC'].dt.to_period('M').dt.to_timestamp()
monthly = df.groupby('year_month').size().reset_index(name='count')

fig = px.line(
    monthly, x='year_month', y='count', markers=True,
    title='📈 Monthly Crime Trend (2020–Present)',
    color_discrete_sequence=[CRIME_RED]
)
fig.update_layout(
    xaxis_title='Month', yaxis_title='Number of Crimes',
    width=950, height=450,
    plot_bgcolor='#0d1117', paper_bgcolor='#0d1117',
    font=dict(color='#cdd6f4'),
    title_font=dict(size=18, color=CRIME_RED)
)
fig.update_xaxes(gridcolor='#2a2a4a')
fig.update_yaxes(gridcolor='#2a2a4a')
fig.show()

In [ ]:
# ── 7.5 Top 15 Crime Types ───────────────────────────────────────────────────
top15 = df['Crm Cd Desc'].value_counts().nlargest(15)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(top15.index[::-1], top15.values[::-1],
               color=[CRIME_RED if i < 5 else CRIME_BLUE for i in range(15)])
ax.set_title('Top 15 Crime Types')
ax.set_xlabel('Count')
for bar in bars:
    ax.text(bar.get_width() + top15.max()*0.005, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9, color='#cdd6f4')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.6 Crimes by Area ───────────────────────────────────────────────────────
area_counts = df['AREA NAME'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
colors_area = [CRIME_RED if v >= area_counts.quantile(0.75) else CRIME_BLUE
               for v in area_counts.values]
ax.bar(area_counts.index, area_counts.values, color=colors_area)
ax.set_title('Crimes by LAPD Area')
ax.set_xlabel('Area')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.7 Victim Demographics ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Victim Demographics', fontsize=16, color=CRIME_RED, y=1.02)

# Age distribution
df_age = df[(df['Vict Age'] > 10) & (df['Vict Age'] < 100)]
axes[0].hist(df_age['Vict Age'], bins=30,
             color=CRIME_RED, edgecolor='#0d1117', alpha=0.85)
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Weapon usage (pie)
weapon_counts = df['Weapon Desc'].value_counts().head(5)
axes[1].pie(weapon_counts, labels=weapon_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=['#e94560','#f7768e','#ff9e64','#e0af68','#9ece6a'],
            textprops={'color': '#cdd6f4', 'fontsize': 8})
axes[1].set_title('Top 5 Weapons Used')

# Descent (pie)
descent_counts = df['Vict Descent'].value_counts().head(5)
axes[2].pie(descent_counts, labels=descent_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=['#7aa2f7','#bb9af7','#2ac3de','#73daca','#b4f9f8'],
            textprops={'color': '#cdd6f4', 'fontsize': 8})
axes[2].set_title('Top 5 Victim Descent')

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.8 Target: Violent vs Non-Violent ───────────────────────────────────────
mapping = {1: 'Violent', 2: 'Non-Violent'}
df['Crime_Type'] = df['Part 1-2'].map(mapping)

counts = df['Crime_Type'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(counts.index, counts.values,
              color=[CRIME_RED, CRIME_BLUE],
              width=0.5, edgecolor='#0d1117', linewidth=1.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
            f'{int(bar.get_height()):,}', ha='center', fontsize=12, fontweight='bold')
ax.set_title('🎯 Target Distribution: Violent vs Non-Violent')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Class balance:\n{(counts / counts.sum() * 100).round(2)}")

<div style="background:#0d1117; border-left:4px solid #f7768e; padding:15px 25px; border-radius:8px;"><h2 style="color:#f7768e; margin:0;">🔢 8 · Encoding Categorical Features</h2></div>

In [ ]:
X_train = X_train.drop('LOCATION', axis=1)
X_test  = X_test.drop('LOCATION', axis=1)

# Encode & save label encoders for top features
obj_cols_saved = ['Crm Cd Desc', 'Vict Descent', 'Weapon Desc', 'Status Desc']
for col in obj_cols_saved:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col]  = le.transform(X_test[col])
    with open(f'Files/{col}_label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)

# Encode remaining object cols (not saved separately)
for col in ['AREA NAME', 'Status']:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col]  = le.transform(X_test[col])

print("✅ Categorical encoding complete")
X_train.dtypes

<div style="background:#0d1117; border-left:4px solid #e0af68; padding:15px 25px; border-radius:8px;"><h2 style="color:#e0af68; margin:0;">🌡️ 9 · Correlation Heatmap</h2></div>

In [ ]:
train_corr = pd.concat([X_train, y_train], axis=1)

fig, ax = plt.subplots(figsize=(22, 12))
mask = np.triu(np.ones_like(train_corr.corr(numeric_only=True), dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(
    train_corr.corr(numeric_only=True),
    annot=True, fmt='.2f', cmap='RdYlBu_r',
    mask=mask, ax=ax,
    annot_kws={'size': 7},
    linewidths=0.5, linecolor='#0d1117'
)
ax.set_title('Correlation Heatmap (Training Set)', pad=20)
plt.tight_layout()
plt.show()

<div style="background:#0d1117; border-left:4px solid #9ece6a; padding:15px 25px; border-radius:8px;"><h2 style="color:#9ece6a; margin:0;">🎯 10 · Feature Selection via Random Forest</h2></div>

In [ ]:
X_train_c = X_train.drop(['Date Rptd', 'DATE OCC'], axis=1)
X_test_c  = X_test.drop(['Date Rptd', 'DATE OCC'], axis=1)

rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train_c, y_train)

importance = pd.Series(rf_selector.feature_importances_, index=X_train_c.columns)
top_features = importance.nlargest(9).index.tolist()
print('Top 9 features:', top_features)

# Visualise feature importance
fig, ax = plt.subplots(figsize=(10, 6))
sorted_imp = importance.nlargest(15).sort_values()
ax.barh(sorted_imp.index, sorted_imp.values,
        color=[CRIME_RED if v >= sorted_imp.quantile(0.6) else CRIME_BLUE
               for v in sorted_imp.values])
ax.set_title('Feature Importance (Random Forest)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# Keep only top features
FEATURES = ['Crm Cd 1', 'Crm Cd Desc', 'Weapon Used Cd', 'Premis Cd',
            'Weapon Desc', 'Vict Age', 'Vict Descent', 'Vict Sex', 'Status Desc']

X_train = X_train[FEATURES]
X_test  = X_test[FEATURES]

print(f"✅ Feature selection done – keeping {len(FEATURES)} features")
X_train.shape

<div style="background:#0d1117; border-left:4px solid #f7768e; padding:15px 25px; border-radius:8px;"><h2 style="color:#f7768e; margin:0;">📐 11 · Outlier Detection & Treatment</h2></div>

In [ ]:
numeric_cols = X_train.columns
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(X_train[col], bins=30, alpha=0.7, color=CRIME_RED,
                 edgecolor='#0d1117', label='Train')
    axes[i].hist(X_test[col], bins=30, alpha=0.5, color=CRIME_BLUE,
                 edgecolor='#0d1117', label='Test')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Feature Distributions – Train vs Test', fontsize=16,
             color=CRIME_RED, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Fix negative Vict Age (treat as missing, re-impute)
X_train.loc[X_train['Vict Age'] < 0, 'Vict Age'] = 0
X_test.loc[X_test['Vict Age']  < 0, 'Vict Age'] = 0

iter_imputer2 = IterativeImputer(missing_values=0, max_iter=15, random_state=42)
X_train[['Vict Age']] = iter_imputer2.fit_transform(X_train[['Vict Age']])
X_test[['Vict Age']]  = iter_imputer2.transform(X_test[['Vict Age']])

with open('Files/Iterative_imputer_Age2.pkl', 'wb') as f:
    pickle.dump(iter_imputer2, f)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(X_train['Vict Age'], bins=40, color=CRIME_RED, edgecolor='#0d1117', alpha=0.85)
ax.set_title('Vict Age Distribution After Outlier Treatment')
ax.set_xlabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
# Skewness report
skew_vals = {col: skew(X_train[col].dropna()) for col in numeric_cols}
skew_df = pd.DataFrame.from_dict(skew_vals, orient='index', columns=['Skewness'])
print("Skewness values:")
print(skew_df.sort_values('Skewness').to_string())

<div style="background:#0d1117; border-left:4px solid #bb9af7; padding:15px 25px; border-radius:8px;"><h2 style="color:#bb9af7; margin:0;">⚖️ 12 · Feature Scaling</h2></div>

In [ ]:
# Encode target
le_y = LabelEncoder()
y_train_enc = le_y.fit_transform(y_train)
y_test_enc  = le_y.transform(y_test)

print("Classes mapping:", dict(zip(le_y.classes_, le_y.transform(le_y.classes_))))
with open('Files/le_y.pkl', 'wb') as f:
    pickle.dump(le_y, f)
print("✅ Target encoded & saved")

In [ ]:
# Scaling groups (based on distribution analysis)
yeo      = ['Weapon Desc', 'Status Desc', 'Vict Sex']
normal   = ['Crm Cd Desc', 'Vict Descent']
log_cols = ['Crm Cd 1', 'Premis Cd']
robust   = ['Vict Age', 'Weapon Used Cd']

# Yeo-Johnson
yeo_t = PowerTransformer(method='yeo-johnson', standardize=True)
X_train[yeo] = yeo_t.fit_transform(X_train[yeo])
X_test[yeo]  = yeo_t.transform(X_test[yeo])
with open('Files/yeo-johnson.pkl', 'wb') as f:
    pickle.dump(yeo_t, f)

# Standard Scaler (near-normal features)
std = StandardScaler()
X_train[normal] = std.fit_transform(X_train[normal])
X_test[normal]  = std.transform(X_test[normal])
with open('Files/StandardScaler.pkl', 'wb') as f:
    pickle.dump(std, f)

# Log transform + Standard Scaler (skewed features)
X_train[log_cols] = np.log1p(X_train[log_cols])
X_test[log_cols]  = np.log1p(X_test[log_cols])
std2 = StandardScaler()
X_train[log_cols] = std2.fit_transform(X_train[log_cols])
X_test[log_cols]  = std2.transform(X_test[log_cols])
with open('Files/StandardScaler2.pkl', 'wb') as f:
    pickle.dump(std2, f)

# Robust Scaler (outlier-prone features)
robust_sc = RobustScaler()
X_train[robust] = robust_sc.fit_transform(X_train[robust])
X_test[robust]  = robust_sc.transform(X_test[robust])
with open('Files/RobustScaler.pkl', 'wb') as f:
    pickle.dump(robust_sc, f)

# Save processed test set
with open('Files/X_test.pkl', 'wb') as f: pickle.dump(X_test, f)
with open('Files/y_test.pkl', 'wb') as f: pickle.dump(y_test, f)

print("✅ All scalers applied and saved")

<div style="background:#0d1117; border-left:4px solid #e94560; padding:15px 25px; border-radius:8px;"><h2 style="color:#e94560; margin:0;">🤖 13 · Model Training & Evaluation</h2></div>

In [ ]:
models = {
    'LogisticRegression':    LogisticRegression(max_iter=1000, solver='lbfgs'),
    'KNeighborsClassifier':  KNeighborsClassifier(n_neighbors=5),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=None, random_state=42),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTreesClassifier':  ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'GaussianNB':            GaussianNB(),
    'Bagging':               BaggingClassifier(
                                 estimator=DecisionTreeClassifier(),
                                 n_estimators=50, random_state=42),
    'AdaBoost':              AdaBoostClassifier(
                                 n_estimators=100, learning_rate=0.5, random_state=42),
    'XGBoost':               XGBClassifier(
                                 n_estimators=200, learning_rate=0.1, max_depth=6,
                                 random_state=42, eval_metric='mlogloss', verbosity=0),
    'LightGBM':              LGBMClassifier(
                                 n_estimators=200, learning_rate=0.1, max_depth=-1,
                                 random_state=42, verbose=-1),
    'CatBoost':              CatBoostClassifier(
                                 iterations=200, learning_rate=0.1, depth=6,
                                 random_state=42, verbose=0),
}
print(f"✅ {len(models)} models ready for training")

In [ ]:
# Train, evaluate & save all models
metrics_list = []
results_dict = {}
class_names  = [str(c) for c in np.unique(y_test_enc)]

for name, model in models.items():
    print(f"⏳ Training {name}...", end=' ')
    model.fit(X_train, y_train_enc)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test_enc, y_pred)
    pre = precision_score(y_test_enc, y_pred, average='binary')
    rec = recall_score(y_test_enc, y_pred, average='binary')
    f1  = f1_score(y_test_enc, y_pred, average='binary')

    metrics_list.append([name, acc, pre, rec, f1])
    results_dict[name] = {
        'metrics': {'Accuracy': acc, 'Precision': pre, 'Recall': rec, 'F1': f1},
        'confusion_matrix': confusion_matrix(y_test_enc, y_pred),
        'classification_report': classification_report(y_test_enc, y_pred, output_dict=True),
        'class_names': class_names
    }

    # Confusion matrix plot
    cm = confusion_matrix(y_test_enc, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
                xticklabels=['Non-Violent', 'Violent'],
                yticklabels=['Non-Violent', 'Violent'], ax=ax,
                linewidths=0.5, linecolor='#0d1117')
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # Save model
    with open(f'Files/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"✅ Accuracy: {acc:.4f}")

# Save results
with open('Files/results_dict.pkl', 'wb') as f:
    pickle.dump(results_dict, f)
print("\n✅ All models saved to Files/")

<div style="background:#0d1117; border-left:4px solid #9ece6a; padding:15px 25px; border-radius:8px;"><h2 style="color:#9ece6a; margin:0;">🏆 14 · Results Summary</h2></div>

In [ ]:
results_df = pd.DataFrame(
    metrics_list, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1']
).sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Style the dataframe
results_df_display = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1']:
    results_df_display[col] = results_df_display[col].map('{:.4f}'.format)

results_df

In [ ]:
# ── Metrics comparison chart ──────────────────────────────────────────────────
fig = go.Figure()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
palette = [CRIME_RED, CRIME_BLUE, ACCENT, '#9ece6a']

for metric, color in zip(metrics, palette):
    fig.add_trace(go.Bar(
        name=metric,
        x=results_df['Model'],
        y=results_df[metric],
        marker_color=color,
        opacity=0.85
    ))

fig.update_layout(
    barmode='group',
    title='📊 Model Performance Comparison',
    xaxis_title='Model',
    yaxis_title='Score',
    yaxis_range=[0.75, 1.02],
    width=1000, height=500,
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='#cdd6f4'),
    title_font=dict(size=18, color='#9ece6a'),
    legend=dict(bgcolor='#1a1a2e', bordercolor='#2a2a4a'),
    xaxis=dict(tickangle=-30)
)
fig.update_xaxes(gridcolor='#2a2a4a')
fig.update_yaxes(gridcolor='#2a2a4a')
fig.show()

<div style="background: linear-gradient(135deg, #0d1117 0%, #1a1a2e 100%); padding: 30px; border-radius: 12px; text-align: center; border: 1px solid #e94560; margin-top: 20px;">
  <h2 style="color:#9ece6a; margin:0 0 10px 0;">✅ Pipeline Complete</h2>
  <p style="color:#a9b1d6; margin:0;">All models trained · Scalers & encoders saved · Results exported to <code style="color:#e94560;">Files/</code></p>
  <hr style="border:1px solid #2a2a4a; margin:15px 0;">
  <p style="color:#565f89; font-size:0.9em; margin:0;">Crime Predictor · LAPD Dataset · 2020–Present</p>
</div>

In [ ]:
print("="*60)
print("  CRIME PREDICTOR — FINAL SUMMARY")
print("="*60)
print(f"  Models trained    : {len(models)}")
print(f"  Best model        : {results_df.iloc[0]['Model']}")
print(f"  Best accuracy     : {float(results_df.iloc[0]['Accuracy']):.4f}")
print(f"  Features used     : {len(FEATURES)}")
print(f"  Files saved       : Files/ directory")
print("="*60)